# Visualization 3: Gender Trends for Selected Names

One visualization only: specific names, male/female evolution over time, and a year cursor.

In [13]:
import altair as alt
import pandas as pd
alt.data_transformers.enable('json')
pd.Series.iteritems = pd.Series.items

## Loading and preparing data

In [14]:
data_dir="../data/"

In [15]:
names = pd.read_csv(f"{data_dir}dpt2020.csv", sep=";")
names = names[(names["preusuel"] != "_PRENOMS_RARES") & (names["dpt"] != "XX")].copy()
names["annais"] = pd.to_numeric(names["annais"], errors="coerce")
names = names.dropna(subset=["annais"])
names["annais"] = names["annais"].astype(int)

selected_names = ["DOMINIQUE", "CAMILLE", "CLAUDE", "ALIX", "CHARLIE", "EDEN"]
trend = names[names["preusuel"].isin(selected_names)].copy()
trend["gender"] = trend["sexe"].map({1: "Male", 2: "Female"})

trend_year = (
    trend.groupby(["annais", "preusuel", "gender"], as_index=False)["nombre"]
    .sum()
    .rename(columns={"annais": "year", "preusuel": "name", "nombre": "births"})
)

trend_year.head()

,year,name,gender,births
0,1900,ALIX,Female,47
1,1900,ALIX,Male,6
2,1900,CAMILLE,Female,711
3,1900,CAMILLE,Male,1188
4,1900,CLAUDE,Male,626


## Multi-line Chart with Year Cursor

Each panel is one name, with male and female trends over time. Use the slider to place the cursor year.

In [17]:
year_slider = alt.binding_range(
    min=int(trend_year["year"].min()),
    max=int(trend_year["year"].max()),
    step=1,
    name="Year: "
)
year_cursor = alt.param(name="year_cursor", value=2000, bind=year_slider)

color_scale = alt.Scale(domain=["Male", "Female"], range=["#4C66AF", "#C4544B"])
base = alt.Chart(trend_year).add_params(year_cursor)

line = base.mark_line(strokeWidth=2).encode(
    x=alt.X("year:Q", title=None, axis=alt.Axis(format="d", tickCount=4, grid=False)),
    y=alt.Y("births:Q", title="Births / year", axis=alt.Axis(grid=False)),
    color=alt.Color("gender:N", title=None, scale=color_scale),
    tooltip=[
        alt.Tooltip("name:N", title="Name"),
        alt.Tooltip("gender:N", title="Gender"),
        alt.Tooltip("year:Q", title="Year", format="d"),
        alt.Tooltip("births:Q", title="Births", format=",")
    ]
)

rule = base.transform_calculate(
    cursor="year_cursor"
).mark_rule(color="firebrick", opacity=0.45, size=2).encode(
    x=alt.X("cursor:Q")
)

points = base.transform_filter(
    alt.datum.year == year_cursor
).mark_point(size=90, filled=True).encode(
    x=alt.X("year:Q"),
    y=alt.Y("births:Q"),
    color=alt.Color("gender:N", title=None, scale=color_scale),
    tooltip=[
        alt.Tooltip("name:N", title="Name"),
        alt.Tooltip("gender:N", title="Gender"),
        alt.Tooltip("year:Q", title="Year", format="d"),
        alt.Tooltip("births:Q", title="Births", format=",")
    ]
)

name_order = ["DOMINIQUE", "CAMILLE", "CLAUDE", "ALIX", "CHARLIE", "EDEN"]

(alt.layer(line, rule, points)
 .properties(width=180, height=140)
 .facet(
     facet=alt.Facet("name:N", sort=name_order, header=alt.Header(title=None, labelFontSize=16)),
     columns=3
 )
 .resolve_scale(y="shared")
 .properties(title="Gender trends per name over time"))

alt.FacetChart(...)